## Project: Summarization

In [67]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

#### A) Basic Prompt

In [68]:
from langchain_groq import ChatGroq
from langchain_classic.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature= 0
)

In [69]:
text ="""
Mean Reciprocal Rank (MRR) is a metric used to evaluate the effectiveness of retrieval systems by measuring how well they rank the first relevant document for a set of queries. It calculates the average of the reciprocal ranks (1 divided by the position of the first relevant result) across all queries. For example, if the first relevant document for a query appears in position 3, the reciprocal rank is 1/3. MRR emphasizes the system’s ability to surface relevant content early in the results, which is critical for applications where users rely on top results, such as search engines or document retrieval in RAG (Retrieval-Augmented Generation) systems.

In the context of a RAG system, MRR helps assess the retriever component’s performance. A RAG system retrieves documents to provide context for a language model to generate answers. If the retriever fails to rank relevant documents highly, the generator may produce inaccurate or irrelevant responses. MRR focuses on the position of the first relevant document, which is particularly useful when the generator depends heavily on the top result. For instance, if a user asks, “What causes climate change?” and the retriever returns a relevant document at position 1, the reciprocal rank is 1. If the first relevant document is at position 4, the reciprocal rank drops to 0.25. Averaging these scores across all test queries gives the MRR, reflecting the retriever’s consistency in prioritizing useful content.

To apply MRR, developers need a labeled dataset where the correct documents for each query are known. Suppose you test three queries:

Query A: First relevant document is at position 1 → RR = 1.
Query B: First relevant document is at position 3 → RR = 1/3.
Query C: No relevant documents in the top 10 → RR = 0. The MRR is (1 + 0.333 + 0) / 3 ≈ 0.444. While MRR is simple to compute, it has limitations. It ignores subsequent relevant documents and doesn’t account for varying query difficulty. For a comprehensive evaluation, combine MRR with metrics like recall@k (which measures how many relevant documents are in the top k results). Improving MRR might involve tuning the retriever’s ranking algorithm, using better embeddings for semantic search, or expanding training data to reduce gaps in retrieval accuracy.
"""

messages = [
    SystemMessage(content = 'You are an expert copywriter with expertise in summarizing documents'),
    HumanMessage(content = f'Please provide a short and concise summary of the following text:\n Text: {text}')
]

In [70]:
summary_output = llm.invoke(messages)
print(summary_output.content)

Here is a concise summary of the text:

Mean Reciprocal Rank (MRR) is a metric that evaluates the effectiveness of retrieval systems by measuring the average position of the first relevant document for a set of queries. It calculates the reciprocal rank (1/position) for each query and averages them. MRR emphasizes the system's ability to surface relevant content early in the results, which is critical for applications like search engines and RAG systems. To apply MRR, a labeled dataset is needed, and it can be used to tune the retriever's ranking algorithm and improve retrieval accuracy. However, MRR has limitations and should be combined with other metrics for a comprehensive evaluation.


### Prompt Template

In [71]:
from langchain_classic import PromptTemplate
from langchain_classic.chains import LLMChain

In [72]:
template = '''
Write a concise and short summary of the following text:
TEXT: `{text}`
Translate the summary to {language}.
'''

prompt = PromptTemplate(
    input_variables=['text', 'language'],
    template=template,
)

In [73]:
chain = prompt | llm
summary = chain.invoke({'text': text, 'language': 'ru'})
print(summary.content)

**English Summary:**
Mean Reciprocal Rank (MRR) is a metric that evaluates the effectiveness of retrieval systems by measuring the rank of the first relevant document for a set of queries. It calculates the average of the reciprocal ranks across all queries, emphasizing the system's ability to surface relevant content early in the results.

**Russian Translation:**
Средний обратный ранг (MRR) — это метрика, которая оценивает эффективность систем поиска, измеряя ранг первого релевантного документа для набора запросов. Она рассчитывает среднее значение обратных рангов для всех запросов, подчеркивая способность системы выводить релевантный контент на ранних позициях результатов.


### Stuffing

In [74]:
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_community.docstore.document import Document

In [75]:
with open('sj.txt') as f:
    text = f.read()
# text

docs = [Document(page_content=text)]

In [76]:
from langchain_classic import PromptTemplate
prompt_template = '''
Write a concise and short summary of the following text:
TEXT: `{text}`
'''
prompt = PromptTemplate(
    input_variables=['text'],
    template=prompt_template,
)

In [77]:
chain = load_summarize_chain(
    llm,
    chain_type= "stuff",
    prompt = prompt,
    verbose=True,
)
output_summary = chain.invoke(docs)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Write a concise and short summary of the following text:
TEXT: `I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I’ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That’s it. No big deal. Just three stories. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born.

My biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute

In [78]:
print(output_summary)

{'input_documents': [Document(metadata={}, page_content='I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I’ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That’s it. No big deal. Just three stories. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born.\n\nMy biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute that they really wanted a girl. So my parents, who were on a waiting list, got a call in the middle of the nigh

### Map Reduce

In [79]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
with open('sj.txt') as f:
    text = f.read()

In [80]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=50
)

chunks = text_splitter.create_documents([text])

In [81]:
len(chunks)

3

In [82]:
chain = load_summarize_chain(
    llm,
    chain_type= "map_reduce",
    verbose=False,
)
output_summary = chain.invoke(chunks)

In [83]:
print(output_summary)

{'input_documents': [Document(metadata={}, page_content='I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I’ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That’s it. No big deal. Just three stories. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born.\n\nMy biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute that they really wanted a girl. So my parents, who were on a waiting list, got a call in the middle of the nigh

In [84]:
chain.llm_chain.prompt.template

'Write a concise summary of the following:\n\n\n"{text}"\n\n\nCONCISE SUMMARY:'

In [85]:
chain.combine_document_chain.llm_chain.prompt.template

'Write a concise summary of the following:\n\n\n"{text}"\n\n\nCONCISE SUMMARY:'

### Map reduce with Custom prompt

In [86]:
map_prompt = '''
Write a short and concise summary of the following text:
TEXT: `{text}`
CONCISE SUMMARY:
'''
map_prompt_template = PromptTemplate(
    input_variables=[text],
    template=map_prompt,
)

In [87]:
combine_prompt = '''
Write a short and concise summary of the following text that cover the key points:
Add title to the summary
Start your summary with INTRODUCTION PARAGRAPH that gives an overview of the topic FOLLOWED BY BULLET POINTS if possible AND end the summary with a CONCLUSION PHRASE.
TEXT: `{text}`
'''
combine_prompt_template = PromptTemplate(
    input_variables=[text],
    template=combine_prompt,
)

In [88]:
summary_chain = load_summarize_chain(
    llm,
    chain_type= "map_reduce",
    map_prompt = map_prompt_template,
    combine_prompt = combine_prompt_template,
    verbose=False,
)
output = summary_chain.invoke(chunks)

In [89]:
print(output)

{'input_documents': [Document(metadata={}, page_content='I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I’ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That’s it. No big deal. Just three stories. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born.\n\nMy biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute that they really wanted a girl. So my parents, who were on a waiting list, got a call in the middle of the nigh

### Refine Combine Document Chain

In [90]:
chain = load_summarize_chain(
    llm,
    chain_type= "refine",
    verbose=False,
)
output_summary = chain.invoke(chunks)

In [91]:
print(output_summary)

{'input_documents': [Document(metadata={}, page_content='I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I’ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That’s it. No big deal. Just three stories. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born.\n\nMy biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute that they really wanted a girl. So my parents, who were on a waiting list, got a call in the middle of the nigh

In [92]:
prompt_template = '''
Write a short and concise summary of the following text:
TEXT: `{text}`
CONCISE SUMMARY:
'''
initial_prompt = PromptTemplate(
    template=prompt_template,
    input_variables=[text],
)
refine_template = '''
Your job is to produce final summary.
I have provided an existing summary up to a certain point: {existing_answer}

Please refine the existing summary with some more context below.
-------------------
{text}
-------------------
Start your summary with INTRODUCTION PARAGRAPH that gives an overview of the topic FOLLOWED BY BULLET POINTS if possible AND end the summary with a CONCLUSION PHRASE.
'''

refine_prompt = PromptTemplate(
    template=refine_template,
    input_variables=['existing_answer','text'],
)

In [93]:
chain = load_summarize_chain(
    llm,
    chain_type= "refine",
    question_prompt = initial_prompt,
    refine_prompt = refine_prompt,
    return_intermediate_steps = False
)
output_summary = chain.invoke(chunks)
print(output_summary)

{'input_documents': [Document(metadata={}, page_content='I am honored to be with you today at your commencement from one of the finest universities in the world. I never graduated from college. Truth be told, this is the closest I’ve ever gotten to a college graduation. Today I want to tell you three stories from my life. That’s it. No big deal. Just three stories. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born.\n\nMy biological mother was a young, unwed college graduate student, and she decided to put me up for adoption. She felt very strongly that I should be adopted by college graduates, so everything was all set for me to be adopted at birth by a lawyer and his wife. Except that when I popped out they decided at the last minute that they really wanted a girl. So my parents, who were on a waiting list, got a call in the middle of the nigh

### Summary using Langchain Agents

In [94]:
from langchain_classic.tools import Tool
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

In [95]:
api_wrapper = WikipediaAPIWrapper(wiki_client=wikipedia)
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper)

In [100]:
tools = [
    Tool(
        name="Wikipedia API",
        func=wikipedia.run,
        description="Useful when you need to look up any topic, country or person to answer any question.",
    ),
]
# 1. Define a manual template that includes the required variables
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

agent = create_react_agent(
    llm,
    tools,
    prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
agent_executor.invoke({"input": "Summarize the career of Steve Jobs using Wikipedia."})



> Entering new AgentExecutor chain...
To summarize the career of Steve Jobs using Wikipedia, I should first look up Steve Jobs on Wikipedia to get an overview of his life and career.

Action: Wikipedia API
Action Input: 'Steve Jobs'